In [ ]:
# ============================================================
# CÉLULA 0 — autenticar somente o client vROps para o probe 15-F.4
# Não executa coleta pesada.
# ============================================================

from pathlib import Path
from datetime import datetime
import pandas as pd

from rmc_vrops_v510 import *

VROPS_HOST = "seu_server.com.br"       # use o mesmo valor do seu notebook
AUTH_SOURCE = "seu_server.com.br"     # use o mesmo authSource que funcionou

OUTDIR = Path("saida_probe_15f4")
LOG_DIR = OUTDIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / f"probe_15f4_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logger = setup_logger(LOG_FILE)

client = VropsClient(VROPS_HOST, auth_source=AUTH_SOURCE, logger=logger)
client.authenticate()

print("Client vROps autenticado.")

In [ ]:
# ============================================================
# PROBE 15-F.4 — descobrir propriedades/statkeys de alocação
# Versão corrigida: carrega HOSTS/VMS do Excel se não existirem
# ============================================================

import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1) Validar client vROps
# ------------------------------------------------------------
if "client" not in globals():
    raise RuntimeError(
        "Variável 'client' não existe. Rode antes as células de autenticação do vROps "
        "no notebook PAC_RMC_vROps_API_Notebook_v5_10_4.ipynb."
    )

# ------------------------------------------------------------
# 2) Carregar HOSTS e VMs se não estiverem no kernel
# ------------------------------------------------------------
xlsx = Path(r"C:\Projetos\rmc_copilot\data\raw\rmc_outputs\RMC_Recursos_VM_v5_10_4_4_3_20260615_130005.xlsx")

if not xlsx.exists():
    xlsx = Path(r"C:\Projetos\rmc_copilot\data\raw\uploads\RMC_Recursos_VM_v5_10_4_4_3_20260615_130005.xlsx")

if not xlsx.exists():
    raise FileNotFoundError(f"Excel não encontrado: {xlsx}")

print("Excel usado:", xlsx)

if "df_hosts" not in globals():
    df_hosts = pd.read_excel(xlsx, sheet_name="HOSTS")
    print("df_hosts carregado:", df_hosts.shape)

if "df_vms" not in globals():
    df_vms = pd.read_excel(xlsx, sheet_name="VMS_SELECIONADAS")
    print("df_vms carregado:", df_vms.shape)

# ------------------------------------------------------------
# 3) Funções auxiliares
# ------------------------------------------------------------
def normalize_statkeys(obj):
    """
    Aceita DataFrame/list/dict retornado por client.get_statkeys()
    e transforma em DataFrame com coluna 'key'.
    """
    if obj is None:
        return pd.DataFrame(columns=["key"])

    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
        if "key" not in df.columns:
            if "statKey" in df.columns:
                df["key"] = df["statKey"]
            elif len(df.columns) == 1:
                df["key"] = df.iloc[:, 0].astype(str)
        return df

    if isinstance(obj, list):
        rows = []
        for item in obj:
            if isinstance(item, dict):
                rows.append(item)
            else:
                rows.append({"key": str(item)})
        df = pd.DataFrame(rows)
        if "key" not in df.columns:
            for c in ["statKey", "name", "id"]:
                if c in df.columns:
                    df["key"] = df[c]
                    break
        return df

    if isinstance(obj, dict):
        # Formatos comuns: {"stat-key": [...]} ou {"statKeys": [...]}
        for k in ["stat-key", "statKeys", "statkeys", "values"]:
            if k in obj:
                return normalize_statkeys(obj[k])
        return pd.DataFrame([obj])

    return pd.DataFrame({"key": [str(obj)]})


def get_properties_raw(client, resource_id):
    data, status, text = client.get_no_raise(
        f"/suite-api/api/resources/{resource_id}/properties",
        params=None,
        timeout=120
    )
    return data, status, text


def print_keys(title, obj, contains=None, max_rows=300):
    print("\n" + "=" * 100)
    print(title)

    df = normalize_statkeys(obj)

    if df is None or df.empty:
        print("VAZIO")
        return

    if "key" not in df.columns:
        print("Sem coluna key. Colunas encontradas:", list(df.columns))
        print(df.head(max_rows).to_string(index=False))
        return

    out = df.copy()

    if contains:
        pattern = "|".join(contains)
        out = out[out["key"].astype(str).str.contains(pattern, case=False, na=False, regex=True)]

    if out.empty:
        print("Nenhuma statkey candidata encontrada.")
    else:
        print(out.head(max_rows).to_string(index=False))


# ------------------------------------------------------------
# 4) Escolher amostras
# ------------------------------------------------------------
# Pode trocar para uma VM específica, se quiser:
VM_SAMPLE = None
# Exemplo:
# VM_SAMPLE = "SRV-DASHPRD01"

if VM_SAMPLE:
    df_vm_match = df_vms[df_vms["vm"].astype(str).str.upper() == VM_SAMPLE.upper()]
    if df_vm_match.empty:
        raise ValueError(f"VM_SAMPLE não encontrada: {VM_SAMPLE}")
    vm_row = df_vm_match.iloc[0]
else:
    vm_row = df_vms.iloc[0]

# Host associado à VM, quando possível
host_name = vm_row.get("mapping_parent_name")

if host_name and "name" in df_hosts.columns:
    df_host_match = df_hosts[df_hosts["name"].astype(str) == str(host_name)]
    if not df_host_match.empty:
        host_row = df_host_match.iloc[0]
    else:
        host_row = df_hosts.iloc[0]
else:
    host_row = df_hosts.iloc[0]

host_id = host_row["identifier"]
vm_id = vm_row["vm_resource_id"]

print("\nHOST AMOSTRA:", host_row.get("name"), host_id)
print("CLUSTER HOST:", host_row.get("cluster"))

print("\nVM AMOSTRA:", vm_row.get("vm"), vm_id)
print("CLUSTER VM:", vm_row.get("cluster"))
print("HOST DA VM:", vm_row.get("mapping_parent_name"))

# ------------------------------------------------------------
# 5) Statkeys HOST
# ------------------------------------------------------------
host_statkeys = client.get_statkeys(host_id)

print_keys(
    "HOST STATKEYS CANDIDATAS",
    host_statkeys,
    contains=[
        "cpu", "core", "cores", "socket", "mem", "memory",
        "capacity", "usable", "demand", "usage", "mhz", "provision"
    ],
    max_rows=300
)

# ------------------------------------------------------------
# 6) Statkeys VM
# ------------------------------------------------------------
vm_statkeys = client.get_statkeys(vm_id)

print_keys(
    "VM STATKEYS CANDIDATAS",
    vm_statkeys,
    contains=[
        "cpu", "vcpu", "numcpu", "core", "mem", "memory",
        "configured", "provision", "capacity", "disk", "datastore"
    ],
    max_rows=300
)

# ------------------------------------------------------------
# 7) Properties HOST
# ------------------------------------------------------------
host_props, host_status, host_text = get_properties_raw(client, host_id)

print("\n" + "=" * 100)
print("HOST PROPERTIES STATUS:", host_status)

if host_props:
    host_json = json.dumps(host_props, ensure_ascii=False, indent=2)
    print(host_json[:20000])
else:
    print(str(host_text)[:20000])

# ------------------------------------------------------------
# 8) Properties VM
# ------------------------------------------------------------
vm_props, vm_status, vm_text = get_properties_raw(client, vm_id)

print("\n" + "=" * 100)
print("VM PROPERTIES STATUS:", vm_status)

if vm_props:
    vm_json = json.dumps(vm_props, ensure_ascii=False, indent=2)
    print(vm_json[:20000])
else:
    print(str(vm_text)[:20000])